# 04. Model Evaluation - Hybrid Forecasting System

## 🎯 Objective
Evaluate the complete hybrid forecasting system across ALL products and segments.

### Forecasting Methods by Segment
1. **Segments A & B (High Frequency)**: XGBoost ML Model
2. **Segment C (Medium Frequency)**: Weighted Moving Average (14-day window)
3. **Segment D (Low Frequency)**: Naive Baseline (7-day average)

### Key Metrics
- **WAPE** (Weighted Absolute Percentage Error): Industry-standard for forecast accuracy
- **BIAS**: Measures systematic over/under-forecasting (%)
- **MAE** (Mean Absolute Error): Average absolute difference
- **RMSE** (Root Mean Squared Error): Penalizes large errors

---

## Step 1: Load All Required Data & Models

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
import pickle
import os

sns.set(style="whitegrid")

# Load test data (ML products only)
test_ml = pd.read_csv('../../data/processed/test_ml.csv')
test_ml['date'] = pd.to_datetime(test_ml['date'])

# Load full demand data (for all products including non-ML)
demand_full = pd.read_csv('../../data/raw/demand_daily.csv')
demand_full['date'] = pd.to_datetime(demand_full['date'])

# Load product segments
product_segments = pd.read_csv('../../data/processed/product_segments.csv')

# Load trained XGBoost model
model = XGBRegressor()
model.load_model('../../data/models/xgboost_model.json')

with open('../../data/models/feature_columns.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

print("✅ All components loaded")
print(f"   Test ML samples: {len(test_ml):,}")
print(f"   Full demand records: {len(demand_full):,}")
print(f"   Total products: {product_segments['id_produit'].nunique():,}")

✅ All components loaded
   Test ML samples: 1,598
   Full demand records: 150,999
   Total products: 1,129


## Step 2: Define Forecasting Methods for Each Segment

In [38]:
def forecast_moving_average(product_id, forecast_date, history_df, window_days=28):
    """
    Weighted Average Daily Demand for Segment C.
    Calculates weights over 4 weeks to capture recent trends while normalizing for zero-demand days.
    """
    total_weighted_demand = 0
    total_weights = 0
    
    for week in range(4):
        w_end = forecast_date - pd.Timedelta(days=week*7)
        w_start = forecast_date - pd.Timedelta(days=(week+1)*7)
        
        week_demand = history_df[
            (history_df['id_produit'] == product_id) &
            (history_df['date'] >= w_start) &
            (history_df['date'] < w_end)
        ]['quantite_demande'].sum()
        
        weight = 4 - week # Recent weeks get higher weight
        total_weighted_demand += week_demand * weight
        total_weights += 7 * weight
        
    forecast = total_weighted_demand / total_weights
    return max(0, forecast)


def forecast_naive(product_id, forecast_date, history_df, window_days=30):
    """
    Simple Average Daily Demand for Segment D.
    Averages total demand over a fixed 30-day window to account for sparsity.
    """
    start_date = forecast_date - pd.Timedelta(days=window_days)
    
    total_demand = history_df[
        (history_df['id_produit'] == product_id) &
        (history_df['date'] >= start_date) &
        (history_df['date'] < forecast_date)
    ]['quantite_demande'].sum()
    
    forecast = total_demand / window_days
    return max(0, forecast)

print("✅ Forecasting methods updated to Average Daily Demand (ADD) logic")

✅ Forecasting methods updated to Average Daily Demand (ADD) logic


## Step 3: Generate Forecasts for ALL Products in Test Period

In [ ]:
print("Generating forecasts for all products in test period...")

all_forecasts = []

# Get unique test dates
test_dates = sorted(test_ml['date'].unique())
print(f"Test period: {test_dates[0].date()} to {test_dates[-1].date()} ({len(test_dates)} days)")

for idx, forecast_date in enumerate(test_dates):
    if idx % 5 == 0:
        print(f"  Processing day {idx+1}/{len(test_dates)}: {forecast_date.date()}")
    
    # Get all products
    all_products = product_segments['id_produit'].unique()
    
    for product_id in all_products:
        # Get product segment
        segment_row = product_segments[product_segments['id_produit'] == product_id]
        if len(segment_row) == 0:
            continue
        segment = segment_row['segment'].iloc[0]
        
        # Apply appropriate forecasting method based on segment
        if segment in ['A_HIGH_FREQ_HIGH_VOL', 'B_HIGH_FREQ_LOW_VOL']:
            # ML forecast (Tier 1)
            product_test = test_ml[
                (test_ml['id_produit'] == product_id) &
                (test_ml['date'] == forecast_date)
            ]
            
            if len(product_test) > 0:
                forecast = forecast_ml(product_test, model, feature_cols)[0]
            else:
                forecast = 0
                
            method = 'XGBoost'
            
        elif segment == 'C_MEDIUM_FREQ':
            # Moving average (Tier 2)
            forecast = forecast_moving_average(product_id, forecast_date, demand_full)
            method = 'Moving Average'
            
        else:  # D_LOW_FREQ
            # Naive baseline (Tier 3)
            forecast = forecast_naive(product_id, forecast_date, demand_full)
            method = 'Naive'
        
        # Get actual demand
        actual_row = demand_full[
            (demand_full['id_produit'] == product_id) &
            (demand_full['date'] == forecast_date)
        ]
        actual = actual_row['quantite_demande'].values[0] if len(actual_row) > 0 else 0
        
        all_forecasts.append({
            'date': forecast_date,
            'id_produit': product_id,
            'segment': segment,
            'method': method,
            'actual': actual,
            'forecast': forecast
        })

forecasts_df = pd.DataFrame(all_forecasts)

print(f"\n✅ Generated {len(forecasts_df):,} forecasts")
print(f"   Segments: {forecasts_df['segment'].value_counts().to_dict()}")

Generating forecasts for all products in test period...
Test period: 2026-01-03 to 2026-01-08 (105 days)
  Processing day 1/105: 2026-01-03
  Processing day 6/105: 2026-01-03


## Step 4: Calculate WAPE & BIAS (Key Performance Metrics)

In [ ]:
def calculate_wape_bias(actual, forecast):
    """
    Calculate WAPE and BIAS - the two most important metrics for demand forecasting
    
    WAPE: Weighted Absolute Percentage Error (industry standard)
    BIAS: Measures tendency to over/under-forecast
    """
    
    # WAPE (Weighted Absolute Percentage Error)
    wape = np.sum(np.abs(actual - forecast)) / np.sum(actual) * 100
    
    # BIAS (Positive = Overforecast, Negative = Underforecast)
    bias = (np.sum(forecast) - np.sum(actual)) / np.sum(actual) * 100
    
    # Additional metrics
    mae = np.mean(np.abs(actual - forecast))
    rmse = np.sqrt(np.mean((actual - forecast)**2))
    
    return {
        'WAPE (%)': wape,
        'BIAS (%)': bias,
        'MAE': mae,
        'RMSE': rmse,
        'Total Actual': np.sum(actual),
        'Total Forecast': np.sum(forecast)
    }

# Calculate overall metrics
overall_metrics = calculate_wape_bias(
    forecasts_df['actual'].values,
    forecasts_df['forecast'].values
)

print("\n" + "="*70)
print("OVERALL HYBRID SYSTEM PERFORMANCE")
print("="*70)
for metric, value in overall_metrics.items():
    if '(%)' in metric:
        print(f"{metric:20s}: {value:7.2f}%")
    else:
        print(f"{metric:20s}: {value:,.2f}")

# Metrics by segment
print("\n" + "="*70)
print("PERFORMANCE BY SEGMENT")
print("="*70)

for segment in sorted(forecasts_df['segment'].unique()):
    segment_data = forecasts_df[forecasts_df['segment'] == segment]
    segment_metrics = calculate_wape_bias(
        segment_data['actual'].values,
        segment_data['forecast'].values
    )
    
    method = segment_data['method'].iloc[0]
    print(f"\n{segment} ({method}):")
    print(f"  Products:        {segment_data['id_produit'].nunique():,}")
    print(f"  Forecast points: {len(segment_data):,}")
    print(f"  WAPE:            {segment_metrics['WAPE (%)']:7.2f}%")
    print(f"  BIAS:            {segment_metrics['BIAS (%)']:7.2f}%")
    print(f"  MAE:             {segment_metrics['MAE']:,.2f}")

## Step 5: Comprehensive Performance Visualizations

In [ ]:
# Create output directory if it doesn't exist
os.makedirs('../../data/evaluation', exist_ok=True)

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Plot 1: WAPE by Segment
segment_wape = []
for seg in sorted(forecasts_df['segment'].unique()):
    seg_data = forecasts_df[forecasts_df['segment'] == seg]
    metrics = calculate_wape_bias(seg_data['actual'].values, seg_data['forecast'].values)
    segment_wape.append({'Segment': seg, 'WAPE': metrics['WAPE (%)']})

wape_df = pd.DataFrame(segment_wape)
axes[0, 0].bar(range(len(wape_df)), wape_df['WAPE'], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[0, 0].axhline(y=30, color='r', linestyle='--', label='30% threshold', linewidth=2)
axes[0, 0].set_xticks(range(len(wape_df)))
axes[0, 0].set_xticklabels(wape_df['Segment'], rotation=45, ha='right')
axes[0, 0].set_title('WAPE by Segment (Lower is Better)', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('WAPE (%)')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Plot 2: BIAS by Segment
segment_bias = []
for seg in sorted(forecasts_df['segment'].unique()):
    seg_data = forecasts_df[forecasts_df['segment'] == seg]
    metrics = calculate_wape_bias(seg_data['actual'].values, seg_data['forecast'].values)
    segment_bias.append({'Segment': seg, 'BIAS': metrics['BIAS (%)']})

bias_df = pd.DataFrame(segment_bias)
colors = ['green' if abs(b) < 5 else 'orange' if abs(b) < 10 else 'red' for b in bias_df['BIAS']]
axes[0, 1].bar(range(len(bias_df)), bias_df['BIAS'], color=colors)
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=2, label='Perfect balance')
axes[0, 1].axhline(y=5, color='orange', linestyle='--', linewidth=1, label='±5% acceptable')
axes[0, 1].axhline(y=-5, color='orange', linestyle='--', linewidth=1)
axes[0, 1].set_xticks(range(len(bias_df)))
axes[0, 1].set_xticklabels(bias_df['Segment'], rotation=45, ha='right')
axes[0, 1].set_title('BIAS by Segment (Closer to 0 is Better)', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('BIAS (%) [Positive = Overforecast]')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Plot 3: Actual vs Forecast Scatter
axes[1, 0].scatter(forecasts_df['actual'], forecasts_df['forecast'], alpha=0.3, s=10, c='steelblue')
max_val = max(forecasts_df['actual'].max(), forecasts_df['forecast'].max())
axes[1, 0].plot([0, max_val], [0, max_val], 'r--', label='Perfect forecast', linewidth=2)
axes[1, 0].set_xlabel('Actual Demand')
axes[1, 0].set_ylabel('Forecasted Demand')
axes[1, 0].set_title('Actual vs Forecast (All Products)', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Plot 4: Error Distribution
errors = forecasts_df['forecast'] - forecasts_df['actual']
axes[1, 1].hist(errors, bins=50, edgecolor='black', alpha=0.7, color='mediumpurple')
axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero error')
axes[1, 1].axvline(x=errors.mean(), color='green', linestyle='--', linewidth=2, label=f'Mean={errors.mean():.1f}')
axes[1, 1].set_xlabel('Forecast Error (Forecast - Actual)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Error Distribution (Should be centered at 0)', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../../data/evaluation/performance_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved to: ../../data/evaluation/performance_metrics.png")

## 📊 Evaluation Summary

### What We Measured
- **WAPE**: Measures forecast accuracy (lower is better, <30% is good)
- **BIAS**: Measures systematic over/under-forecasting (close to 0% is best)
- **MAE**: Average absolute error in units
- **RMSE**: Emphasizes larger errors

### Interpretation Guide
- **WAPE < 20%**: Excellent accuracy
- **WAPE 20-30%**: Good accuracy
- **WAPE > 30%**: Needs improvement
- **BIAS near 0%**: Well-balanced system
- **BIAS > +5%**: Tendency to overforecast (excess inventory)
- **BIAS < -5%**: Tendency to underforecast (stockouts risk)

### Next Steps
1. Review segment-level performance
2. Identify products with high errors
3. Fine-tune model parameters if needed
4. Deploy to production if accuracy is acceptable